# train_fc_v1 corpus EDA

Initial exploratory view of `data/train_fc_v1.parquet` before any model work.

**Corpus shape on disk:** 177,976 path-rows (8 paths/bulk × ~22,245 bulks). Parquet is untouched; filtering happens at load time only.

**Per-row schema (39 cols):**
- 10 metadata: `sample_id, path_idx, qap_name, fe_provenance, h2o_provenance, fc_status, T_FC_end_attrition` + 3 condition cols below
- 11 bulk oxide wt% (model inputs `X`): `sio2, al2o3, cao, mgo, feo, fe2o3, k2o, na2o, tio2, cr2o3, h2o_tot`
- 3 FC conditions (training-time supervision for CondHead, NOT model inputs): `T_FC_end_used, P_kbar, h2o_path_wt`
- 18 mineral wt% (modal targets `Y`): `cpx, ol, q, spl, bi, fsp, ilm, opx, fl, amp, liq, sph, ru, cd, g, mu, ep, ne`

**Load-time filters** (corpus on disk is untouched):
1. drop `fc_status != "ok"` — 69 rows with MAGEMin convergence failures
2. drop rows where `liq > 10 wt%` — physically real FC endpoints but reintroduce v0's melt-inflation problem if trained on. Concentrated in foidite-family classes (depressed wet solidus, can't reach <10% melt within MAGEMin's calibrated T range) and the fractionation-trapped gabbro/diorite tail.

**Things to look at:**
- Class balance — natural prevalence in Gard et al. 2019, basalt-dominated. Affects stratification.
- Condition distribution — what (T, P, H2O) ranges CondHead will see at training.
- Phase presence — rare phases (<1% row presence) will be hard to fit; ~14 of 18 phases expected above that threshold.
- Mass sum sanity — should be ~1.0 per row; large deviations indicate FC output schema bug.
- Audit flag rate per class — `T_FC_end_attrition` is per-bulk, set during pre-pass at canonical H2O. The path-level reality (the actual `liq` value, used by our filter) is the real signal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# notebook expects to be run from notebooks/ subdirectory, or from repo root
DATA = Path('../data/train_fc_v1.parquet')
if not DATA.exists():
    DATA = Path('data/train_fc_v1.parquet')

df = pd.read_parquet(DATA)
print(f'corpus on disk: {df.shape}')
print(f'  ok rows:      {(df.fc_status == "ok").sum():,}')
print(f'  fc_errors:    {df.fc_status.str.startswith("fc_error").sum():,}')
print(f'  unique bulks: {df.sample_id.nunique():,}')
print(f'  classes:      {df.qap_name.nunique()}')

# load-time filter — corpus on disk is untouched
LIQ_THRESHOLD = 0.10
ok = df[(df.fc_status == 'ok') & (df.liq <= LIQ_THRESHOLD)].copy().reset_index(drop=True)

n_high_liq = ((df.fc_status == 'ok') & (df.liq > LIQ_THRESHOLD)).sum()
print(f'\nafter load-time filter (fc_status==ok AND liq <= {LIQ_THRESHOLD*100:.0f}%):')
print(f'  rows kept:           {len(ok):,} ({100*len(ok)/len(df):.1f}% of corpus)')
print(f'  rows dropped (high liq): {n_high_liq:,}')
print(f'  mean liq:            {ok.liq.mean():.4f}  (v0 baseline was 0.70)')
print(f'  median liq:          {ok.liq.median():.4f}')
print(f'  unique bulks left:   {ok.sample_id.nunique():,}')

META_COLS = ['sample_id', 'path_idx', 'qap_name', 'fe_provenance',
             'h2o_provenance', 'fc_status', 'T_FC_end_attrition']
OXIDE_COLS = ['sio2', 'al2o3', 'cao', 'mgo', 'feo', 'fe2o3',
              'k2o', 'na2o', 'tio2', 'cr2o3', 'h2o_tot']
COND_COLS = ['T_FC_end_used', 'P_kbar', 'h2o_path_wt']
PHASE_COLS = [c for c in df.columns
              if c not in META_COLS and c not in OXIDE_COLS and c not in COND_COLS]
print(f'\nphases ({len(PHASE_COLS)}): {PHASE_COLS}')

## headline metric: mean liq mass fraction

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ok['liq'].hist(bins=80, ax=ax1, color='steelblue', alpha=0.8)
ax1.axvline(0.05, color='r', linestyle='--', label='5% threshold')
ax1.axvline(ok['liq'].mean(), color='g', linestyle='--',
            label=f'corpus mean = {ok["liq"].mean():.4f}')
ax1.set_xlabel('liq mass fraction')
ax1.set_ylabel('row count')
ax1.set_title(f'liq across {len(ok):,} rows  (v0 baseline mean was 0.70)')
ax1.legend()

# per-class boxplot, top + bottom 8 by median
med = ok.groupby('qap_name')['liq'].median().sort_values()
show = list(med.head(8).index) + list(med.tail(8).index)
sub = ok[ok.qap_name.isin(show)]
order = med[med.index.isin(show)].index.tolist()
data = [sub[sub.qap_name == c]['liq'].values for c in order]
ax2.boxplot(data, labels=order, showfliers=False)
ax2.axhline(0.05, color='r', linestyle='--', lw=0.8)
ax2.set_xlabel('rock class')
ax2.set_ylabel('liq mass frac')
ax2.set_title('liq per class (lowest 8 + highest 8 by median)')
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

## class distribution

natural prevalence in Gard et al. 2019; basalt-dominated. Worth knowing for stratification and per-class evaluation later.

In [ ]:
vc = ok['qap_name'].value_counts()
print(f'{len(vc)} unique classes, top-20 covering {100*vc.head(20).sum()/len(ok):.1f}% of rows')

fig, ax = plt.subplots(figsize=(12, 4))
vc.head(20).plot.bar(ax=ax, color='steelblue')
ax.set_ylabel('path-rows')
ax.set_xlabel('qap_name')
ax.set_title('top-20 classes by row count')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

## conditions distribution (CondHead targets)

These three quantities are what the CondHead has to predict from bulk alone in stage 2. Their marginal distributions tell us the difficulty:
- T: bimodal (felsic ~650, mafic-ultramafic ~700-1000) — should be predictable from bulk
- P: discrete-ish (class-conditional values from tp_priors), should be moderately predictable
- H2O: continuous, depends on class H2O range × per-path sampling

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, label in zip(axes,
                          ['T_FC_end_used', 'P_kbar', 'h2o_path_wt'],
                          ['T_FC_end (C)', 'P (kbar)', 'H2O (wt%)']):
    ok[col].hist(bins=40, ax=ax, color='steelblue', alpha=0.8)
    ax.set_xlabel(label)
    ax.set_ylabel('row count')
    ax.set_title(f'{label}  median={ok[col].median():.1f}')
plt.tight_layout()
plt.show()

In [ ]:
# T_FC_end_used per top-12 class
top12 = ok['qap_name'].value_counts().head(12).index.tolist()
sub = ok[ok.qap_name.isin(top12)]

fig, ax = plt.subplots(figsize=(12, 4))
data = [sub[sub.qap_name == c]['T_FC_end_used'].values for c in top12]
ax.boxplot(data, labels=top12, showfliers=False)
ax.set_xlabel('rock class')
ax.set_ylabel('T_FC_end (C)')
ax.set_title('T_FC_end per class (top-12 by row count)')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

## phase presence + per-class modal

Phases with < 1% presence are hard to learn; v0 had 14 of 17 phases > 1% reach R² > 0.5. Expect roughly similar threshold here.

In [ ]:
presence = (ok[PHASE_COLS] > 0.01).mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
presence.plot.bar(ax=ax, color='steelblue')
ax.axhline(0.01, color='r', linestyle='--', lw=0.8, label='1% threshold (hard to learn below)')
ax.set_ylabel('row fraction where phase > 1 wt%')
ax.set_title(f'phase presence across {len(ok):,} rows')
ax.legend()
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('phase presence:')
for p, v in presence.items():
    print(f'  {p:6s} {v:.3f}')

In [ ]:
# mean modal per top-12 class
top12 = ok['qap_name'].value_counts().head(12).index.tolist()
means = ok[ok.qap_name.isin(top12)].groupby('qap_name')[PHASE_COLS].mean()
means = means.reindex(top12)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(means.values, aspect='auto', cmap='viridis', vmin=0)
ax.set_xticks(range(len(PHASE_COLS)))
ax.set_xticklabels(PHASE_COLS, rotation=45, ha='right')
ax.set_yticks(range(len(top12)))
ax.set_yticklabels(means.index)
plt.colorbar(im, ax=ax, label='mean mass fraction')
ax.set_title('mean modal vector per class (top-12)')
plt.tight_layout()
plt.show()

## sanity: mass sum per row

Should be ~1.0 since the FC extractor normalized cumsum solids + residual liq + fl per row.  
Large deviations would indicate an FC output schema bug.

In [ ]:
mass_sums = ok[PHASE_COLS].sum(axis=1)
fig, ax = plt.subplots(figsize=(10, 4))
mass_sums.hist(bins=60, ax=ax, color='steelblue', alpha=0.8)
ax.axvline(1.0, color='r', linestyle='--', label='target 1.0')
ax.set_xlabel('row-wise sum of mineral wt%')
ax.set_ylabel('row count')
ax.set_title(f'mass sum: mean={mass_sums.mean():.4f}, std={mass_sums.std():.4f}, '
             f'min={mass_sums.min():.4f}, max={mass_sums.max():.4f}')
ax.legend()
plt.tight_layout()
plt.show()

## bulk oxide distribution by class

The 11 oxide wt% are the model inputs. Distinct class clusters in oxide space are what makes the inverse-inference problem learnable: the model's encoder has to internally separate the rock types based on these features.

In [ ]:
top8 = ok['qap_name'].value_counts().head(8).index.tolist()
sub = ok[ok.qap_name.isin(top8)]

fig, axes = plt.subplots(3, 4, figsize=(15, 9))
for i, ox in enumerate(OXIDE_COLS):
    ax = axes[i // 4, i % 4]
    data = [sub[sub.qap_name == c][ox].values for c in top8]
    ax.boxplot(data, labels=top8, showfliers=False)
    ax.set_title(ox)
    ax.set_ylabel('wt%')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
# hide unused subplot
axes[2, 3].axis('off')
plt.tight_layout()
plt.show()

## audit-flag and provenance breakdown

Bulks where the per-bulk T_FC_end pre-pass couldn't get liq < 5% are flagged. Per-class flag rates tell us which classes have fractionation-trapped melt issues.

In [ ]:
# attrition flag per class
flag_rate = ok.groupby('qap_name')['T_FC_end_attrition'].mean().sort_values(ascending=False)
flag_rate_top = flag_rate.head(15)

fig, ax = plt.subplots(figsize=(10, 4))
flag_rate_top.plot.bar(ax=ax, color='coral')
ax.set_ylabel('fraction of rows flagged')
ax.set_title('T_FC_end_attrition rate per class (top 15)')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f'\ntotal flag rate: {ok["T_FC_end_attrition"].mean():.3f}')
print(f'\nfe provenance counts:')
print(ok['fe_provenance'].value_counts())
print(f'\nh2o provenance counts:')
print(ok['h2o_provenance'].value_counts())

## rare-class drop audit

Picking the threshold for `min_bulks_per_class` — the corpus-level rule that decides which qap_name classes survive the split into train/val/test.

**Principle:** every retained class needs at least ~5 bulks in each split. At a 20% test split, that's a floor of 25 bulks per class. Classes below the chosen threshold are **dropped from the corpus entirely** (not held in train), so we can compute honest per-class metrics on test.

This is destructive: we're saying we won't evaluate the model on those rock types. The cells below help judge what we'd lose.

In [ ]:
per_bulk = ok.drop_duplicates(subset=['sample_id']).reset_index(drop=True)
counts = per_bulk['qap_name'].value_counts()

# summary at each candidate threshold
print(f'corpus baseline: {len(per_bulk)} bulks, {len(counts)} classes, {len(ok)} rows\n')
print(f'{"threshold":>10}  {"classes":>7}  {"bulks":>7}  {"rows":>8}  {"pct_corpus":>10}')
print(f'{"-"*48}')
for N in [5, 10, 15, 25, 50, 75, 100]:
    kept = counts[counts >= N]
    n_kept_bulks = int(kept.sum())
    n_kept_rows = int(ok[ok.qap_name.isin(kept.index)].shape[0])
    pct = 100 * n_kept_rows / len(ok)
    star = '  *' if N == 25 else ''
    print(f'{N:>10}  {len(kept):>7}  {n_kept_bulks:>7}  {n_kept_rows:>8}  {pct:>9.2f}%{star}')
print('  (* = recommended default: every retained class has at least ~5 in 20% test split)')

# bar chart of class counts with reference lines
THRESHOLD = 25
fig, ax = plt.subplots(figsize=(14, 4))
order = counts.sort_values(ascending=False).index.tolist()
colors = ['steelblue' if counts[c] >= THRESHOLD else 'lightcoral' for c in order]
ax.bar(range(len(order)), counts.loc[order].values, color=colors, edgecolor='none', width=0.85)
ax.set_yscale('log')
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=75, ha='right', fontsize=7)
ax.set_ylabel('bulks per class (log scale)')
ax.set_title(f'class counts: blue = kept at threshold {THRESHOLD}, red = dropped')

for ref, label, color in [(5, 'n=5', 'gray'), (25, 'n=25 (recommended)', 'k'), (50, 'n=50', 'gray')]:
    ax.axhline(ref, color=color, linestyle='--', lw=0.8, alpha=0.8)
    ax.text(len(order) - 0.5, ref * 1.15, label, ha='right', va='bottom',
            fontsize=8, color=color)
plt.tight_layout()
plt.show()

print(f'\nclasses dropped at threshold {THRESHOLD}:')
dropped = counts[counts < THRESHOLD].sort_values()
for name, n in dropped.items():
    print(f'  {n:3d}  {name}')

In [ ]:
# chemistry isolation check: TAS-style scatter (SiO2 vs Na2O+K2O)
# Are dropped classes in chemistry regions the kept classes don't cover?

THRESHOLD = 25
kept_classes = set(counts[counts >= THRESHOLD].index)
dropped_classes = list(counts[counts < THRESHOLD].sort_values().index)

per_bulk_chem = per_bulk[['sample_id', 'qap_name', 'sio2', 'na2o', 'k2o']].copy()
per_bulk_chem['alkali'] = per_bulk_chem['na2o'] + per_bulk_chem['k2o']

kept_chem = per_bulk_chem[per_bulk_chem.qap_name.isin(kept_classes)]
dropped_chem = per_bulk_chem[per_bulk_chem.qap_name.isin(dropped_classes)]

fig, ax = plt.subplots(figsize=(11, 6))
hb = ax.hexbin(kept_chem.sio2, kept_chem.alkali, gridsize=40, cmap='Blues',
               mincnt=1, alpha=0.7, edgecolors='none')
plt.colorbar(hb, ax=ax, label='kept-class bulks per hex')

# Mean position per dropped class with bulk count label
import itertools
markers = ['o', 's', '^', 'D', 'v', '<', '>', 'P', 'X', '*']
cmap_drop = plt.cm.tab20(np.linspace(0, 1, len(dropped_classes)))
for i, cls in enumerate(dropped_classes):
    sub = dropped_chem[dropped_chem.qap_name == cls]
    mx, my = sub.sio2.mean(), sub.alkali.mean()
    ax.scatter(mx, my, s=80, color=cmap_drop[i],
               marker=markers[i % len(markers)],
               edgecolor='black', linewidth=0.7, zorder=5)
    ax.annotate(f'{cls} (n={len(sub)})', (mx, my), fontsize=7,
                xytext=(4, 4), textcoords='offset points', alpha=0.85)

ax.set_xlabel('SiO2 wt%')
ax.set_ylabel('Na2O + K2O wt%')
ax.set_title(f'TAS-style: kept classes (hexbin background) vs dropped at threshold {THRESHOLD}\n'
             f'dropped points inside the hexbin density = safe to drop (similar chemistry to kept)')
plt.tight_layout()
plt.show()

In [ ]:
# per-class oxide means for borderline classes (< 50 bulks)
# Highlight cells whose value is far from corpus mean.

borderline_classes = counts[counts < 50].sort_values(ascending=False).index.tolist()
borderline_means = (per_bulk[per_bulk.qap_name.isin(borderline_classes)]
                    .groupby('qap_name')[OXIDE_COLS].mean()
                    .reindex(borderline_classes))
corpus_mean = per_bulk[OXIDE_COLS].mean().rename('corpus_mean')
tbl = pd.concat([borderline_means, corpus_mean.to_frame().T], axis=0)
tbl['n_bulks'] = [counts[c] for c in borderline_classes] + [len(per_bulk)]
tbl = tbl[['n_bulks'] + OXIDE_COLS]

# pickled for later reuse if the threshold decision needs revisiting
tbl.to_parquet('../data/rare_class_means.parquet')

tbl.style.background_gradient(subset=OXIDE_COLS, cmap='RdBu_r', axis=0).format('{:.2f}', subset=OXIDE_COLS).format('{:.0f}', subset=['n_bulks'])

### threshold decision

- chosen `min_bulks_per_class` =
- rationale:

(record here once the bar chart, TAS scatter, and oxide-mean table have been inspected; this value flows into `make_loaders_v1_bulk_split` in `src/cnmineral/models/training.py`)